# Benchmark JAX GPU Colab — PR #14b (Sprint 5.1b)

**Objetivo**: medir performance do simulador Python otimizado em **GPU T4/A100** via Colab Pro+ nos 3 caminhos JAX:

1. **JAX híbrido** (`use_native_dipoles=False`) — via `pure_callback` → Numba host (CPU dentro do nó GPU)
2. **JAX nativo forward** (`use_native_dipoles=True`) — XLA fusion total
3. **JAX jacfwd nativo** — `jax.jacfwd` sobre `forward_pure_jax`

Referência de CPU (macOS Intel Core i9 16 threads, medido localmente):
- Numba: 3,6 ms/modelo (oklahoma_3)
- Fortran tatu.x: 29,7 ms/modelo
- JAX nativo CPU: ~17.000 ms/modelo (JIT overhead dominante sem batching)

**Este notebook deve ser executado manualmente no Colab com runtime T4 ou A100.**

## 0. Setup Colab

In [ ]:
# Instala dependências (rode em célula separada no Colab)
!pip install -q numpy 'numba>=0.60' 'jax[cuda12]>=0.4.30'

# Clona o repo (ajuste a branch conforme necessário)
!git clone -b main https://github.com/daniel-guitarplayer-8/geosteering-ai.git /content/Geosteering_AI
%cd /content/Geosteering_AI

In [ ]:
import os, sys, time
os.environ['JAX_ENABLE_X64'] = 'True'
sys.path.insert(0, '/content/Geosteering_AI')

import jax
jax.config.update('jax_enable_x64', True)
print('JAX devices:', jax.devices())
print('JAX default backend:', jax.default_backend())
print('JAX version:', jax.__version__)

## 1. Forward JAX nativo — modelos canônicos

In [ ]:
import numpy as np
from geosteering_ai.simulation._jax.forward_pure import (
    build_static_context, forward_pure_jax,
)
from geosteering_ai.simulation.validation.canonical_models import get_canonical_model

def bench_forward_jax_native(model_name: str, n_pos: int = 100, n_iter: int = 5):
    m = get_canonical_model(model_name)
    positions_z = np.linspace(m.min_depth - 2.0, m.max_depth + 2.0, n_pos)
    ctx = build_static_context(
        m.rho_h, m.rho_v, m.esp, positions_z,
        np.array([20000.0]), 1.0, 0.0,
    )
    # Warmup JIT (primeiro trace é lento)
    H = forward_pure_jax(ctx.rho_h_jnp, ctx.rho_v_jnp, ctx)
    H.block_until_ready()
    t0 = time.perf_counter()
    for _ in range(n_iter):
        H = forward_pure_jax(ctx.rho_h_jnp, ctx.rho_v_jnp, ctx)
        H.block_until_ready()
    elapsed = (time.perf_counter() - t0) / n_iter
    return elapsed, np.asarray(H).shape

print(f'{"Modelo":<20} {"n_pos":>6} {"T_forward (ms)":>16} {"throughput":>14}')
for name in ['oklahoma_3', 'oklahoma_5', 'oklahoma_28', 'hou_7', 'viking_graben_10']:
    t, shape = bench_forward_jax_native(name, n_pos=100, n_iter=5)
    throughput = 3600.0 / t
    print(f'{name:<20} {shape[0]:>6d} {t*1000:>15.2f} {throughput:>12.0f} mod/h')

## 2. Jacobiano jax.jacfwd end-to-end nativo em GPU

In [ ]:
def bench_jacfwd_gpu(model_name: str, n_pos: int = 50, n_iter: int = 3):
    m = get_canonical_model(model_name)
    positions_z = np.linspace(m.min_depth - 2.0, m.max_depth + 2.0, n_pos)
    ctx = build_static_context(
        m.rho_h, m.rho_v, m.esp, positions_z,
        np.array([20000.0]), 1.0, 0.0,
    )
    def fwd(rh, rv):
        return forward_pure_jax(rh, rv, ctx)
    jacfn = jax.jit(jax.jacfwd(fwd, argnums=(0, 1)))
    # Warmup
    J_h, J_v = jacfn(ctx.rho_h_jnp, ctx.rho_v_jnp)
    J_h.block_until_ready()
    t0 = time.perf_counter()
    for _ in range(n_iter):
        J_h, J_v = jacfn(ctx.rho_h_jnp, ctx.rho_v_jnp)
        J_h.block_until_ready()
    elapsed = (time.perf_counter() - t0) / n_iter
    return elapsed, J_h.shape

print(f'{"Modelo":<20} {"n_layers":>8} {"T_jacfwd (ms)":>16}')
for name in ['oklahoma_3', 'oklahoma_5', 'hou_7']:
    t, shape = bench_jacfwd_gpu(name, n_pos=50, n_iter=3)
    print(f'{name:<20} {shape[-1]:>8d} {t*1000:>15.2f}')

## 3. Heatmap ∂H/∂ρ_h (oklahoma_5)

In [ ]:
import matplotlib.pyplot as plt

m = get_canonical_model('oklahoma_5')
positions_z = np.linspace(m.min_depth - 2.0, m.max_depth + 2.0, 80)
ctx = build_static_context(
    m.rho_h, m.rho_v, m.esp, positions_z,
    np.array([20000.0]), 1.0, 0.0,
)
J_h, J_v = jax.jit(jax.jacfwd(lambda rh, rv: forward_pure_jax(rh, rv, ctx), argnums=(0,1)))(
    ctx.rho_h_jnp, ctx.rho_v_jnp
)
J_h_np = np.asarray(J_h)  # (n_pos, 1, 9, n_layers)
print('J_h shape:', J_h_np.shape)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
im0 = axes[0].imshow(np.abs(J_h_np[:, 0, 0, :]), aspect='auto', cmap='viridis')
axes[0].set_title('|∂Hxx/∂ρ_h| (oklahoma_5)')
axes[0].set_xlabel('camada')
axes[0].set_ylabel('posição z')
plt.colorbar(im0, ax=axes[0])
im1 = axes[1].imshow(np.abs(J_h_np[:, 0, 8, :]), aspect='auto', cmap='magma')
axes[1].set_title('|∂Hzz/∂ρ_h| (oklahoma_5)')
axes[1].set_xlabel('camada')
plt.colorbar(im1, ax=axes[1])
plt.tight_layout()
plt.show()

## 4. Comparação GPU T4 vs CPU local (referência)

Preencha esta tabela após executar as células acima no Colab:

| Modelo | CPU Intel i9 Numba (ms) | CPU Intel i9 JAX nativo (ms) | GPU T4 JAX nativo (ms) | Speedup GPU/CPU nativo |
|:-------|-----------------------:|----------------------------:|----------------------:|----------------------:|
| oklahoma_3 | 3,61 | 17.135 | (executar) | (calcular) |
| oklahoma_5 | 3,92 | 17.111 | (executar) | (calcular) |
| oklahoma_28 | 7,17 | 17.359 | (executar) | (calcular) |

**Expectativa**: GPU T4 com `jax.jit` deve obter speedup de 10×–50× sobre o caminho nativo CPU (XLA fusion + paralelismo massivo dos SMs). Numba CPU permanece a opção mais rápida em CPU puro para até ~1000 modelos.